# Entrenamiento de locomoción Unitree G1 — LocoMuJoCo (PFI) — Versión LOCAL

Pensado para correr en tu compu, sin GPU, con entrenamiento dividido en **etapas idempotentes** que permiten retomar el progreso si el proceso se corta.

## Antes de abrir este notebook

Corré esto en una terminal, **antes** de abrir Jupyter (no son celdas de este notebook, son pasos previos):

```bash
python3 -m venv venv_g1
source venv_g1/bin/activate          # Windows: venv_g1\Scripts\activate

pip install jax
pip install -U flax
pip install "git+https://github.com/robfiras/loco-mujoco.git@master"
pip install jupyterlab

jupyter lab
```

Abrí este notebook desde esa sesión de Jupyter (para que use el `venv_g1` con todo instalado).

## Qué garantiza este notebook y qué no
- **Sí garantiza:** si el proceso se corta a mitad de una etapa de entrenamiento (corte de luz, cierre accidental, crash), volver a correr la celda de entrenamiento detecta las etapas ya completas y sigue solo desde la que faltaba — como máximo perdés el trabajo de la etapa en curso.
- **No hace:** retomar desde el step exacto donde se cortó dentro de una misma etapa (la librería no lo soporta de forma nativa; investigarlo a fondo implicaría modificar código interno de LocoMuJoCo, con riesgo de más problemas de compatibilidad). Por eso conviene elegir etapas de duración razonable para tu máquina (ver sección 5).

## 1. Confirmar el entorno (versiones, CPU disponible)

In [1]:
import jax, flax, numpy, os
print('jax:', jax.__version__)
print('flax:', flax.__version__)
print('numpy:', numpy.__version__)
print('Dispositivos JAX:', jax.devices())  # esperado: CpuDevice, no CudaDevice
print('Núcleos de CPU disponibles:', os.cpu_count())


jax: 0.10.2
flax: 0.12.8
numpy: 2.4.6
Dispositivos JAX: [CpuDevice(id=0)]
Núcleos de CPU disponibles: 16


## 2. Definir carpetas de proyecto (local, sin Google Drive)

In [2]:
import os

PROJECT_DIR = os.path.expanduser('~/PFI_G1_locomotion')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
VIDEO_DIR = os.path.join(PROJECT_DIR, 'videos')
STATE_FILE = os.path.join(PROJECT_DIR, 'training_state.json')

for d in [PROJECT_DIR, CHECKPOINT_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

print('Carpeta de proyecto:', PROJECT_DIR)


Carpeta de proyecto: C:\Users\chris/PFI_G1_locomotion


## 3. Clonar el repositorio (para los scripts de entrenamiento)

In [3]:
%cd ~
!git clone --depth 1 https://github.com/robfiras/loco-mujoco.git
%cd ~/loco-mujoco
!ls examples/training_examples
%cd ~


C:\Users\chris
C:\Users\chris\loco-mujoco


fatal: destination path 'loco-mujoco' already exists and is not an empty directory.


C:\Users\chris


"ls" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## 4. Verificación de imports y creación del entorno del G1

Local no necesita el orden especial de imports que usábamos en Colab para esquivar el import circular del paquete de PyPI roto -- pero como igual instalamos desde GitHub (mismo código), lo dejamos por las dudas, no cuesta nada.

In [4]:
import loco_mujoco
from loco_mujoco.environments import LocoEnv
print('LocoEnv importado OK')

from loco_mujoco.task_factories import ImitationFactory, LAFAN1DatasetConf
print('task_factories importado OK')

print([n for n in LocoEnv.registered_envs.keys() if 'G1' in n or 'Unitree' in n])




LocoEnv importado OK
task_factories importado OK
['UnitreeH1', 'MjxUnitreeH1', 'UnitreeH1v2', 'MjxUnitreeH1v2', 'UnitreeG1', 'MjxUnitreeG1', 'UnitreeA1', 'MjxUnitreeA1', 'UnitreeGo2', 'MjxUnitreeGo2']


## 5. Entrenamiento por etapas idempotentes (el corazón de este notebook)

`STAGES` es la lista de metas de steps totales, de menor a mayor. Cada etapa entrena **desde cero** hasta esa meta (la librería no soporta continuar desde una etapa anterior), pero si el checkpoint de una etapa ya existe en disco, esa etapa se saltea automáticamente.

**Ajustá `STAGES` a tu paciencia y a la velocidad real de tu CPU.** Sin GPU, este tipo de entrenamiento con JAX suele ser 20-50x más lento que en GPU -- los números de acá abajo son un punto de partida conservador, no una recomendación firme. Corré primero la etapa más chica sola y medí cuánto tarda antes de comprometerte a las etapas grandes.

`num_envs` se ajusta automáticamente según los núcleos de tu CPU (ver celda anterior), pero podés fijarlo a mano si preferís.

In [7]:
import os, json, glob, shutil, time

STAGES = [1_000_000, 5_000_000, 20_000_000, 50_000_000, 100_000_000]  # ajustar según tu CPU , 5_000_000, 20_000_000, 50_000_000, 100_000_000

NUM_ENVS = min(32, max(4, os.cpu_count() or 8))
print('num_envs a usar:', NUM_ENVS)

os.environ['WANDB_MODE'] = 'disabled'
os.environ['HYDRA_FULL_ERROR'] = '1'

def stage_checkpoint_path(steps):
    return os.path.join(CHECKPOINT_DIR, f'g1_walk_{steps}steps.pkl')

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return {'completed_stages': []}

def save_state(state):
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)

state = load_state()
print('Etapas ya completas según el estado guardado:', state['completed_stages'])


num_envs a usar: 16
Etapas ya completas según el estado guardado: [1000000]


In [ ]:
%cd ~/loco-mujoco/examples/training_examples/jax_rl_mimic

for steps in STAGES:
    ckpt_path = stage_checkpoint_path(steps)

    if steps in state['completed_stages'] and os.path.exists(ckpt_path):
        print(f'Etapa {steps} steps ya completa (checkpoint encontrado), saltando...')
        continue

    print(f'=== Entrenando etapa: {steps} steps ===')
    t0 = time.time()

    exit_code = os.system(
        f'python experiment.py '
        f'experiment.env_params.env_name=MjxUnitreeG1 '
        f'experiment.env_params.use_mjwarp=false '
        f'experiment.task_factory.params.lafan1_dataset_conf.dataset_name=[walk1_subject1] '
        f'experiment.total_timesteps={steps} '
        f'experiment.num_envs={NUM_ENVS}'
    )

    elapsed = time.time() - t0
    print(f'Etapa {steps} steps terminada en {elapsed/60:.1f} min. Código de salida: {exit_code}')

    if exit_code != 0:
        print(f'ADVERTENCIA: la etapa {steps} terminó con error. Revisá el output antes de seguir.')
        break

    pkl_files = glob.glob('./outputs/**/*.pkl', recursive=True)
    if not pkl_files:
        print(f'ADVERTENCIA: no se encontró .pkl para la etapa {steps}. Revisar antes de continuar.')
        break

    latest = max(pkl_files, key=os.path.getmtime)
    shutil.copy(latest, ckpt_path)
    print('Checkpoint guardado en:', ckpt_path)

    state['completed_stages'].append(steps)
    save_state(state)

print('Loop de entrenamiento por etapas terminado. Etapas completas:', state['completed_stages'])


C:\Users\chris\loco-mujoco\examples\training_examples\jax_rl_mimic
Etapa 1000000 steps ya completa (checkpoint encontrado), saltando...
=== Entrenando etapa: 5000000 steps ===
Etapa 5000000 steps terminada en 54.0 min. Código de salida: 0
Checkpoint guardado en: C:\Users\chris/PFI_G1_locomotion\checkpoints\g1_walk_5000000steps.pkl
=== Entrenando etapa: 20000000 steps ===
Etapa 20000000 steps terminada en 833.5 min. Código de salida: 0
Checkpoint guardado en: C:\Users\chris/PFI_G1_locomotion\checkpoints\g1_walk_20000000steps.pkl
=== Entrenando etapa: 50000000 steps ===


**Si el proceso se corta acá** (cerraste el notebook, se colgó la máquina, lo que sea): simplemente volvé a abrir este notebook, corré las celdas 1 a 4 de nuevo (setup), y volvé a correr esta misma celda de la sección 5 tal cual. El `state['completed_stages']` leído desde `training_state.json` va a hacer que salte automáticamente las etapas ya guardadas y continúe desde donde quedó.

## 6. Evaluación del checkpoint de mayor etapa completada

In [ ]:
state = load_state()
if not state['completed_stages']:
    print('Todavía no hay ninguna etapa completa.')
else:
    best_steps = max(state['completed_stages'])
    best_ckpt = stage_checkpoint_path(best_steps)
    print(f'Mejor checkpoint disponible: {best_ckpt} ({best_steps} steps)')


In [ ]:
%cd ~/loco-mujoco/examples/training_examples/jax_rl_mimic
!python eval.py --help


In [ ]:
eval_video_path = os.path.join(VIDEO_DIR, f'eval_{best_steps}steps.mp4')

# Ajustar los flags según el --help de arriba si no coinciden exactamente
!python eval.py --path "$best_ckpt"

# Nota: corriendo local con pantalla real, render=True (usado internamente por eval.py)
# debería abrir una ventana o grabar sin el lío de Xvfb/EGL que tuvimos en Colab.
# Si estás en un servidor remoto sin pantalla, ahí sí hace falta Xvfb -- avisame si es el caso.


## 7. Integración a tu proyecto de GitHub

1. El checkpoint de mayor etapa completada está en `CHECKPOINT_DIR` (`~/PFI_G1_locomotion/checkpoints/`).
2. Copialo a tu repo:
   ```
   /models/g1_walk_policy/g1_walk_XXXXXXXXsteps.pkl
   /src/inference/run_policy.py
   ```
3. Si pesa mucho, Git LFS:
   ```bash
   git lfs install
   git lfs track "models/g1_walk_policy/**"
   git add .gitattributes models/g1_walk_policy
   git commit -m "Agregar policy de locomoción entrenada (G1, DeepMimic)"
   ```
4. Para cargar el checkpoint en tu `launch.py` o script de inferencia, necesitamos confirmar la firma real de `PPOJax.load_agent` y cómo extraer la función de la red desde `agent_conf`/`agent_state` para aplicarla en un loop de control -- eso lo resolvemos en la próxima conversación cuando tengas el checkpoint final, revisando el código real en vez de asumir nombres de métodos.